In [1]:
# bfe
import pandas as pd
import os
import os.path as osp
import numpy as np
import json
from datetime import datetime
from zoneinfo import ZoneInfo
import requests

In [2]:

try:
    PROJECT_ROOT = osp.abspath(osp.join(osp.dirname(__file__),'..'))
except:
    PROJECT_ROOT = osp.abspath(osp.join(os.getcwd(),'..'))

print(PROJECT_ROOT)
STATIONS_FILE = osp.join(PROJECT_ROOT,'src','stations_infos.json')

/home/bfeldmann/projects/formation_mlops/projet_meteoAUS


In [4]:
dict_stations = {
    'Canberra':'IDCJDW2801',
    'Adelaide':'IDCJDW5081',
    'Albany':'IDCJDW6001',
    'Albury':'IDCJDW2002',
    'AliceSprings':'IDCJDW8002',
    'BadgerysCreek':'',
    'Ballarat':'IDCJDW3005',
    'Bendigo':'IDCJDW3008',
    'Brisbane':'IDCJDW4019',
    'Cairns':'IDCJDW4024',
    'Cobar':'IDCJDW2029',
    'CoffsHarbour':'IDCJDW2030',
    'Dartmoor':'IDCJDW3101',
    'Darwin':'IDCJDW8014',
    'GoldCoast':'IDCJDW4050',
    'Hobart':'IDCJDW7021',
    'Katherine':'IDCJDW8024',
    'Launceston':'IDCJDW7025',
    'Melbourne':'IDCJDW3033',
    'MelbourneAirport':'IDCJDW3049',
    'Mildura':'IDCJDW3051',
    'Moree':'IDCJDW2084',
    'MountGambier':'IDCJDW5041',
    'MountGinini':'IDCJDW2804',
    'Newcastle':'IDCJDW2098',
    'Nhil':'IDCJDW3059',
    'NorahHead':'IDCJDW2099',
    'NorfolkIsland':'IDCJDW2100',
    'Nuriootpa':'IDCJDW5049',
    'PearceRAAF':'',
    'Penrith':'IDCJDW2111',
    'Perth':'IDCJDW6111',
    'PerthAirport':'IDCJDW6110',
    'Portland':'IDCJDW3068',
    'Richmond':'IDCJDW2119',
    'Sale':'IDCJDW3022',
    'SalmonGums':'IDCJDW6119',
    'Sydney':'IDCJDW2124',
    'SydneyAirport':'IDCJDW2125',
    'Townsville':'IDCJDW4128',
    'Tuggeranong':'IDCJDW2802',
    'Uluru':'',
    'WaggaWagga':'IDCJDW2139',
    'Walpole':'IDCJDW6138',
    'Watsonia':'IDCJDW3079',
    'Williamtown':'IDCJDW2145',
    'Witchcliffe':'IDCJDW6081',
    'Wollongong':'IDCJDW2146',
    'Woomera':'IDCJDW5072'
}

In [19]:
url_daily_weather = 'https://www.bom.gov.au/climate/dwo/{year_month}/text/{bom_id}.{year_month}.csv'

csv_url = url_daily_weather.format(year_month='202604',bom_id=dict_stations['Canberra'])
filepath = osp.join(PROJECT_ROOT,'raw_data',osp.basename(csv_url))

# now = datetime.now(ZoneInfo("Australia/Melbourne"))
raw_cols = ['Date', 'Minimum temperature (°C)', 'Maximum temperature (°C)',
       'Rainfall (mm)', 'Evaporation (mm)', 'Sunshine (hours)',
       'Direction of maximum wind gust ', 'Speed of maximum wind gust (km/h)',
       'Time of maximum wind gust', '9am Temperature (°C)',
       '9am relative humidity (%)', '9am cloud amount (oktas)',
       '9am wind direction', '9am wind speed (km/h)', '9am MSL pressure (hPa)',
       '3pm Temperature (°C)', '3pm relative humidity (%)',
       '3pm cloud amount (oktas)', '3pm wind direction',
       '3pm wind speed (km/h)', '3pm MSL pressure (hPa)']

dict_cols_map = {
    'Date':'O', 'MinTemp': 'float32', 'MaxTemp': 'float32', 'Rainfall': 'float32', 'Evaporation': 'float32',
    'Sunshine': 'float32', 'WindGustDir': 'string', 'WindGustSpeed': 'float32', 'TimeMaxWindGust': 'string',
    'Temp9am': 'float32', 'Humidity9am': 'int16', 'Cloud9am': 'float32', 'WindDir9am': 'string',
    'WindSpeed9am': 'float32', 'Pressure9am': 'float32', 'Temp3pm': 'float32', 'Humidity3pm': 'int16',
    'Cloud3pm': 'float32', 'WindDir3pm': 'string', 'WindSpeed3pm': 'float32', 'Pressure3pm': 'float32'}
df = pd.read_csv(filepath, delimiter=',', skiprows=7, encoding='latin-1')
df = df.drop(columns=df.columns[0])
# df = df.drop(columns='Time of maximum wind gust')

if all(df.columns == raw_cols):
    df.columns = dict_cols_map.keys()
else:
    raise ValueError('Columns names invalid')

for col,dtype in dict_cols_map.items():
    if dtype in ('int16', 'float32'):
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(dtype)
    elif dtype=='datetime':
        df[col] = pd.to_datetime(df[col], errors='coerce')
    else:
        df[col] = df[col].astype(dtype)

df = df.replace(
    [np.nan, None, "NaN", "nan", "NAN", "NA", "N/A", "", " ", "null", "None"],
    pd.NA
)
df


,Date,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,TimeMaxWindGust,Temp9am,...,Cloud9am,WindDir9am,WindSpeed9am,Pressure9am,Temp3pm,Humidity3pm,Cloud3pm,WindDir3pm,WindSpeed3pm,Pressure3pm
0,2026-04-1,6.3,26.700001,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,12.800000,...,<NA>,SE,9.0,1023.900024,25.900000,32,<NA>,NW,17.0,1022.099976
1,2026-04-2,9.5,<NA>,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,14.500000,...,<NA>,N,6.0,1025.099976,27.400000,32,<NA>,NW,19.0,1021.799988
2,2026-04-3,7.6,25.0,0.0,<NA>,<NA>,E,41.0,16:58,13.600000,...,<NA>,<NA>,<NA>,1023.900024,24.400000,38,<NA>,NE,13.0,1022.000000
3,2026-04-4,11.6,17.299999,0.0,<NA>,<NA>,ENE,28.0,17:12,13.200000,...,8.0,NE,9.0,1027.199951,15.800000,67,8.0,NE,11.0,1025.900024
4,2026-04-5,5.2,20.0,0.0,<NA>,<NA>,ENE,30.0,17:52,13.800000,...,8.0,SSE,6.0,1027.800049,17.600000,57,3.0,NNE,11.0,1023.799988
5,2026-04-6,4.6,17.299999,0.0,<NA>,<NA>,NW,20.0,21:15,12.000000,...,8.0,SSW,9.0,1021.500000,16.799999,81,8.0,W,9.0,1016.799988
6,2026-04-7,6.9,25.200001,4.0,<NA>,<NA>,WNW,48.0,13:12,17.200001,...,<NA>,NNW,9.0,1013.700012,24.299999,36,<NA>,WNW,30.0,1011.099976
7,2026-04-8,7.9,24.200001,0.0,<NA>,<NA>,NNW,39.0,13:47,16.100000,...,<NA>,E,4.0,1017.400024,23.799999,25,<NA>,W,24.0,1015.299988
8,2026-04-9,7.9,26.6,0.0,<NA>,<NA>,WSW,59.0,18:53,18.000000,...,<NA>,<NA>,<NA>,1018.900024,25.200001,44,7.0,NW,26.0,1015.000000
9,2026-04-10,15.5,21.700001,7.6,<NA>,<NA>,NW,63.0,14:23,18.100000,...,8.0,NW,39.0,1012.799988,20.500000,65,8.0,NW,43.0,1009.799988


In [ ]:
class DailyWeatherDATA:
    def __init__(self):
        self.url_daily_weather = 'https://www.bom.gov.au/climate/dwo/{year_month}/text/{bom_id}.{year_month}.csv'

        with open(STATIONS_FILE) as src:
            self.stations_infos = json.load(src)
            
    def get_url(self,city, time='last'):
        station_info = self.stations_infos.get(city,None)
        
        if station_info is None:
            raise ValueError(f"Unknown city: {city}")
        
        if time=='last':
            now = datetime.now(ZoneInfo(station_info['timezone']))
            date_month = now.strftime("%Y%m")
        elif len(time)==6 and time[0:2]=='20':
            date_month = time
        else:
            raise ValueError(f'Not valid time, expected "yearmonth" but got {time}')
        
        return self.url_daily_weather.format(year_month=date_month, bom_id=station_info['bom_id'])
    
    def get_report(self,csv_url, out_path):
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(csv_url, headers=headers)
        response.raise_for_status()

        with open(out_path, "wb") as f:
            f.write(response.content)
        return True

    def cleaning_report(self,in_file, out_file):
        raw_cols = ['Date', 'Minimum temperature (°C)', 'Maximum temperature (°C)',
                    'Rainfall (mm)', 'Evaporation (mm)', 'Sunshine (hours)',
                    'Direction of maximum wind gust ', 'Speed of maximum wind gust (km/h)',
                    'Time of maximum wind gust', '9am Temperature (°C)',
                    '9am relative humidity (%)', '9am cloud amount (oktas)',
                    '9am wind direction', '9am wind speed (km/h)', '9am MSL pressure (hPa)',
                    '3pm Temperature (°C)', '3pm relative humidity (%)',
                    '3pm cloud amount (oktas)', '3pm wind direction',
                    '3pm wind speed (km/h)', '3pm MSL pressure (hPa)']

        dict_cols = {
            'Date':'datetime', 'MinTemp': 'float32', 'MaxTemp': 'float32', 'Rainfall': 'float32', 'Evaporation': 'float32',
            'Sunshine': 'float32', 'WindGustDir': 'string', 'WindGustSpeed': 'float32', 'TimeMaxWindGust': 'string',
            'Temp9am': 'float32', 'Humidity9am': 'int16', 'Cloud9am': 'float32', 'WindDir9am': 'string',
            'WindSpeed9am': 'float32', 'Pressure9am': 'float32', 'Temp3pm': 'float32', 'Humidity3pm': 'int16',
            'Cloud3pm': 'float32', 'WindDir3pm': 'string', 'WindSpeed3pm': 'float32', 'Pressure3pm': 'float32'
            }
        
        df = pd.read_csv(in_file, delimiter=',', skiprows=7, encoding='latin-1')
        df = df.drop(columns=df.columns[0])
        
        if all(df.columns == raw_cols):
            df.columns = dict_cols.keys()
            
        for col,dtype in dict_cols.items():
            if dtype in ('int16', 'float32'):
                df[col] = pd.to_numeric(df[col], errors='coerce').astype(dtype)
            elif dtype=='datetime':
                df[col] = pd.to_datetime(df[col], errors='coerce')
            else:
                df[col] = df[col].astype(dtype)
        
        df = df.replace(
            [np.nan, None, "NaN", "nan", "NAN", "NA", "N/A", "", " ", "null", "None"],
            pd.NA
        )
        df = df.drop(columns='TimeMaxWindGust')        
        df.to_csv(out_file)

In [7]:
data_file = osp.join(PROJECT_ROOT,'data','weatherAUS.csv')
df = pd.read_csv(data_file)
np.unique(df['Location'])

array(['Adelaide', 'Albany', 'Albury', 'AliceSprings', 'BadgerysCreek',
       'Ballarat', 'Bendigo', 'Brisbane', 'Cairns', 'Canberra', 'Cobar',
       'CoffsHarbour', 'Dartmoor', 'Darwin', 'GoldCoast', 'Hobart',
       'Katherine', 'Launceston', 'Melbourne', 'MelbourneAirport',
       'Mildura', 'Moree', 'MountGambier', 'MountGinini', 'Newcastle',
       'Nhil', 'NorahHead', 'NorfolkIsland', 'Nuriootpa', 'PearceRAAF',
       'Penrith', 'Perth', 'PerthAirport', 'Portland', 'Richmond', 'Sale',
       'SalmonGums', 'Sydney', 'SydneyAirport', 'Townsville',
       'Tuggeranong', 'Uluru', 'WaggaWagga', 'Walpole', 'Watsonia',
       'Williamtown', 'Witchcliffe', 'Wollongong', 'Woomera'],
      dtype=object)